# Лабораторная работа №1

## Компьютерная лексикография, терминологические базы (TBX) и Translation Memory (TMX) в Smartcat

**ФИО студента:** Асадова Хадижа Эльдаровна
**Группа:** ИНФА-221
**Вариант:** 1 (предметная область: Информационные технологии: разработка ПО и DevOps)

---

**Цель:** научиться программно обрабатывать стандарты обмена терминологическими
данными (TBX) и памятью переводов (TMX), реализовать поиск нечётких совпадений
(Fuzzy Match) и подстановку терминов глоссария, экспортировать результат в
CAT-систему Smartcat.

**Порядок выполнения:**

1. Выполните ячейки Блока 1 (установка библиотек) и Блока 2 (данные).
2. Реализуйте четыре функции в Блоке 4 вместо маркеров `# TODO`.
3. Запустите Блок 5 — все `assert` должны пройти без ошибок.
4. Соберите файлы своего варианта, экспортируйте CSV, оформите отчёт.

**Оценивание (10 баллов):** парсинг TBX — 2, парсинг TMX — 2, Fuzzy Match — 3,
интеграция глоссария — 2, оформление — 1.

## Блок 1. Setup — установка библиотек

In [23]:
!pip install -q Levenshtein pandas lxml

In [24]:
from __future__ import annotations

import os
import re
import unicodedata
from typing import Any, Dict, List, Optional, Sequence, Tuple

import pandas as pd

try:
    from lxml import etree as ET

    LXML_AVAILABLE = True
except ImportError:
    import xml.etree.ElementTree as ET

    LXML_AVAILABLE = False

try:
    import Levenshtein as _lev

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна между двумя строками."""
        return _lev.distance(a, b)

except ImportError:

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна (резервная реализация на чистом Python)."""
        if a == b:
            return 0
        if not a:
            return len(b)
        if not b:
            return len(a)
        previous = list(range(len(b) + 1))
        for i, char_a in enumerate(a, start=1):
            current = [i]
            for j, char_b in enumerate(b, start=1):
                current.append(min(previous[j] + 1,
                                   current[j - 1] + 1,
                                   previous[j - 1] + (char_a != char_b)))
            previous = current
        return previous[-1]


# ВАЖНО: атрибут xml:lang в парсере раскрывается в полное имя с пространством имён
XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"

pd.set_option("display.max_colwidth", 60)
print("Окружение готово. XML backend:", "lxml" if LXML_AVAILABLE else "ElementTree")

Окружение готово. XML backend: lxml


## Блок 2. Теория: структура стандартов

### 2.1. TBX (TermBase eXchange, ISO 30042)

Формат обмена терминологическими базами. Организован **по понятиям**
(concept-oriented): одна `termEntry` = одно понятие, внутри — по одной `langSet`
на каждый язык.

```xml
<martif type="TBX" xml:lang="en">
  <martifHeader>…</martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Предметная область</descrip>
      <langSet xml:lang="en">
        <tig>
          <term>flight controller</term>
          <termNote type="partOfSpeech">noun</termNote>
        </tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp>
          <term>полётный контроллер</term>
        </termGrp></ntig>
      </langSet>
    </termEntry>
  </body></text>
</martif>
```

| Тег | Назначение |
|-----|------------|
| `martif` | корень документа TBX-Core v2 |
| `termEntry` | терминологическая статья (одно понятие) |
| `langSet` | языковая секция, язык задаётся атрибутом `xml:lang` |
| `tig` / `ntig` | компактная / расширенная группа термина |
| `term` | сам термин |
| `termNote` | грамматическая или иная помета (`partOfSpeech`, `usage`) |
| `descrip` | описание: `definition`, `subjectField`, `context` |

### 2.2. TMX (Translation Memory eXchange, версия 1.4b)

Формат обмена памятью переводов. Организован **по сегментам**.

```xml
<tmx version="1.4">
  <header srclang="en" segtype="sentence" datatype="plaintext"
          creationtool="…" adminlang="en" o-tmf="TMX"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Aerospace</prop>
      <tuv xml:lang="en"><seg>The autopilot holds the altitude.</seg></tuv>
      <tuv xml:lang="ru"><seg>Автопилот выдерживает высоту.</seg></tuv>
    </tu>
  </body>
</tmx>
```

| Тег | Назначение |
|-----|------------|
| `header` | метаданные памяти: инструмент, язык оригинала, тип сегментации |
| `tu` | translation unit — единица перевода |
| `tuv` | translation unit variant — вариант на одном языке (`xml:lang`) |
| `seg` | текст сегмента |
| `prop` | произвольное свойство (`domain`, `client`, `project`) |

### 2.3. Типы совпадений в CAT-системах

| Совпадение | Диапазон | Действие переводчика |
|------------|----------|----------------------|
| ICE (101 %) | 100 % + совпал контекст | подстановка без правки |
| Exact | 100 % | проверить контекст |
| High Fuzzy | 95–99 % | косметическая правка |
| Fuzzy | 75–94 % | редактирование |
| No Match | < 75 % | перевод с нуля |

## Блок 3. Данные

Ячейка ниже создаёт **демонстрационные** файлы `demo.tbx` и `demo.tmx` — они
нужны только для автотестов из Блока 5. Файлы своего варианта
(`variant_NN.tbx` — не менее 15 терминов, `variant_NN.tmx` — не менее 10
предложений) вы создаёте самостоятельно: либо аналогичным кодом, либо выгрузкой
из Smartcat, либо вручную.

In [25]:
DEMO_TBX = """<?xml version="1.0" encoding="UTF-8"?>
<martif type="TBX" xml:lang="en">
  <martifHeader><fileDesc><titleStmt><title>Demo glossary</title></titleStmt>
  </fileDesc></martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en">
        <tig><term>flight controller</term>
        <termNote type="partOfSpeech">noun</termNote>
        <descrip type="definition">Onboard flight computer.</descrip></tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp><term>полётный контроллер</term></termGrp></ntig>
      </langSet>
    </termEntry>
    <termEntry id="tid-002">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>payload</term>
        <termNote type="partOfSpeech">noun</termNote></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>целевая нагрузка</term>
        </termGrp></ntig></langSet>
    </termEntry>
    <termEntry id="tid-003">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>telemetry link</term></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>канал телеметрии</term>
        </termGrp></ntig></langSet>
    </termEntry>
  </body></text>
</martif>
"""

DEMO_TMX = """<?xml version="1.0" encoding="UTF-8"?>
<tmx version="1.4">
  <header creationtool="demo" creationtoolversion="1.0" segtype="sentence"
          o-tmf="TMX" adminlang="en" srclang="en" datatype="plaintext"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The flight controller stabilises the aircraft.</seg></tuv>
      <tuv xml:lang="ru"><seg>Полётный контроллер стабилизирует аппарат.</seg></tuv>
    </tu>
    <tu tuid="2">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The operator checks the telemetry link.</seg></tuv>
      <tuv xml:lang="ru"><seg>Оператор проверяет канал телеметрии.</seg></tuv>
    </tu>
    <tu tuid="3">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The payload is mounted under the fuselage.</seg></tuv>
      <tuv xml:lang="ru"><seg>Целевая нагрузка устанавливается под фюзеляжем.</seg></tuv>
    </tu>
  </body>
</tmx>
"""

with open("demo.tbx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TBX)
with open("demo.tmx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TMX)

print("Созданы файлы:", [f for f in os.listdir(".") if f.endswith((".tbx", ".tmx"))])

Созданы файлы: ['demo.tmx', 'demo.tbx', 'variant_1.tbx', 'variant_1.tmx']


## Блок 4. Задание — реализуйте четыре функции

Требования ко всем функциям: типизация из `typing`, соответствие PEP 8,
обработка отсутствующих элементов без падения (`None`-проверки).

**Подсказки:**

* язык секции читается как `element.get(XML_LANG)`, а не `element.get("lang")`;
* обход `<term>` внутри `langSet` удобно делать через `lang_set.iter("term")` —
  это покрывает и `tig`, и `ntig/termGrp`;
* текст сегмента лучше собирать через `"".join(seg.itertext())` — так не
  теряется текст при наличии инлайновых тегов;
* при подстановке терминов сортируйте их по убыванию длины.

In [26]:
def parse_tbx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TBX-глоссарий в плоскую таблицу терминологических пар.

    Args:
        file_path: путь к ``.tbx``-файлу.
        src_lang: код языка исходных терминов (значение ``xml:lang``).
        tgt_lang: код языка переводных эквивалентов.

    Returns:
        ``pd.DataFrame`` с колонками:
        ``entry_id`` (str)      — значение атрибута ``id`` у ``termEntry``;
        ``source_term`` (str)   — термин на языке ``src_lang``;
        ``target_term`` (str)   — термин на языке ``tgt_lang``;
        ``pos`` (str)           — часть речи из ``termNote[@type='partOfSpeech']``;
        ``definition`` (str)    — текст ``descrip[@type='definition']`` или "";
        ``subject_field`` (str) — текст ``descrip[@type='subjectField']`` или "".
        Статьи, в которых нет одного из двух языков, пропускаются.

    Raises:
        FileNotFoundError: если файл не существует.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    # --- безопасный разбор XML (без DTD и внешних сущностей) ---
    if LXML_AVAILABLE:
        parser = ET.XMLParser(resolve_entities=False, load_dtd=False, no_network=True)
        root = ET.parse(file_path, parser).getroot()
    else:
        root = ET.parse(file_path).getroot()

    rows: List[Dict[str, str]] = []

    for entry in root.iter("termEntry"):
        subject = ""
        descrip = entry.find("descrip[@type='subjectField']")
        if descrip is not None and descrip.text:
            subject = descrip.text.strip()

        by_lang: Dict[str, Dict[str, str]] = {}
        for lang_set in entry.findall("langSet"):
            lang = (lang_set.get(XML_LANG) or "").split("-")[0].lower()
            if not lang:
                continue

            # iter("term") покрывает и <tig>, и <ntig><termGrp>
            terms = [t.text.strip() for t in lang_set.iter("term") if (t.text or "").strip()]
            if not terms:
                continue

            note = lang_set.find(".//termNote[@type='partOfSpeech']")
            definition = lang_set.find(".//descrip[@type='definition']")

            by_lang[lang] = {
                "term": terms[0],
                "pos": (note.text or "").strip() if note is not None else "",
                "definition": (definition.text or "").strip() if definition is not None else "",
            }

        if src_lang in by_lang and tgt_lang in by_lang:
            rows.append({
                "entry_id": entry.get("id", ""),
                "source_term": by_lang[src_lang]["term"],
                "target_term": by_lang[tgt_lang]["term"],
                "pos": by_lang[src_lang]["pos"] or by_lang[tgt_lang]["pos"],
                "definition": by_lang[src_lang]["definition"],
                "subject_field": subject,
            })

    return pd.DataFrame(
        rows,
        columns=["entry_id", "source_term", "target_term",
                 "pos", "definition", "subject_field"],
    )


def parse_tmx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TMX-память переводов в таблицу параллельных сегментов.

    Args:
        file_path: путь к ``.tmx``-файлу.
        src_lang: код исходного языка.
        tgt_lang: код языка перевода.

    Returns:
        ``pd.DataFrame`` с колонками:
        ``tuid`` (str)            — идентификатор единицы перевода;
        ``source_segment`` (str)  — текст ``seg`` на языке ``src_lang``;
        ``target_segment`` (str)  — текст ``seg`` на языке ``tgt_lang``;
        ``domain`` (str)          — текст ``prop[@type='domain']`` или "";
        ``src_len`` (int)         — длина исходного сегмента в символах;
        ``prev_source`` (str)     — предыдущий исходный сегмент (для ICE), для
        первой строки — пустая строка.

    Raises:
        FileNotFoundError: если файл не существует.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    if LXML_AVAILABLE:
        parser = ET.XMLParser(resolve_entities=False, load_dtd=False, no_network=True)
        root = ET.parse(file_path, parser).getroot()
    else:
        root = ET.parse(file_path).getroot()

    def _seg_text(tuv) -> str:
        """Достать текст сегмента, игнорируя инлайновые теги TMX (ph/bpt/ept/it)."""
        seg = tuv.find("seg")
        if seg is None:
            return ""
        return re.sub(r"\s+", " ", "".join(seg.itertext())).strip()

    rows: List[Dict[str, Any]] = []

    for order, tu in enumerate(root.iter("tu"), start=1):
        segments: Dict[str, str] = {}
        for tuv in tu.findall("tuv"):
            lang = (tuv.get(XML_LANG) or tuv.get("lang") or "").split("-")[0].lower()
            text = _seg_text(tuv)
            if lang and text:
                segments[lang] = text

        if src_lang not in segments or tgt_lang not in segments:
            continue

        domain_node = tu.find("prop[@type='domain']")
        rows.append({
            "tuid": tu.get("tuid", str(order)),
            "source_segment": segments[src_lang],
            "target_segment": segments[tgt_lang],
            "domain": (domain_node.text or "").strip() if domain_node is not None else "",
            "src_len": len(segments[src_lang]),
        })

    frame = pd.DataFrame(
        rows,
        columns=["tuid", "source_segment", "target_segment", "domain", "src_len"],
    )
    frame["prev_source"] = frame["source_segment"].shift(1).fillna("")
    return frame


def calculate_fuzzy_match(source_query: str, tm_df: pd.DataFrame,
                          threshold: float = 75.0,
                          context_before: Optional[str] = None,
                          top_n: int = 5) -> List[Dict[str, Any]]:
    """Найти в памяти переводов сегменты, похожие на запрос.

    Алгоритм: нормализовать строки (нижний регистр, без пунктуации), посчитать
    процент сходства по Левенштейну и Token Sort Ratio, взять максимум,
    отбросить кандидатов ниже ``threshold``, отсортировать по убыванию.

    Args:
        source_query: сегмент, который нужно перевести.
        tm_df: таблица памяти переводов (результат :func:`parse_tmx`).
        threshold: минимальный процент сходства (0–100).
        context_before: предыдущий сегмент документа; если он совпадает с
            ``prev_source`` кандидата, а сходство равно 100 %, тип совпадения —
            ICE.
        top_n: максимальное число возвращаемых кандидатов.

    Returns:
        Список словарей, отсортированный по убыванию ``score``, каждый со
        ключами: ``tuid``, ``source_segment``, ``target_segment``,
        ``score`` (float, 0–100), ``lev_ratio``, ``token_sort_ratio``,
        ``match_type`` (одно из ``"ICE (101%)"``, ``"Exact (100%)"``,
        ``"High Fuzzy"``, ``"Fuzzy"``, ``"Low Fuzzy"``).
        Пустой список, если совпадений выше порога нет.
    """
    if tm_df is None or tm_df.empty or not source_query:
        return []

    punct_re = re.compile(r"[^\w\s]", re.UNICODE)

    def _normalize(text: str) -> str:
        """NFKC, нижний регистр, ё→е, без пунктуации, одиночные пробелы."""
        text = unicodedata.normalize("NFKC", text).lower().replace("ё", "е")
        text = punct_re.sub(" ", text)
        return re.sub(r"\s+", " ", text).strip()

    def _lev_ratio(a: str, b: str) -> float:
        """Процент сходства по Левенштейну (0–100)."""
        if not a and not b:
            return 100.0
        longest = max(len(a), len(b))
        return round((1.0 - edit_distance(a, b) / longest) * 100.0, 2)

    def _token_sort_ratio(a: str, b: str) -> float:
        """Процент сходства при игнорировании порядка слов."""
        sa = " ".join(sorted(_normalize(a).split()))
        sb = " ".join(sorted(_normalize(b).split()))
        return _lev_ratio(sa, sb)

    def _classify(score: float, context_matched: bool) -> str:
        """Тип совпадения по проценту и признаку совпадения контекста."""
        if score >= 100.0:
            return "ICE (101%)" if context_matched else "Exact (100%)"
        if score >= 95.0:
            return "High Fuzzy"
        if score >= 75.0:
            return "Fuzzy"
        return "Low Fuzzy"

    results: List[Dict[str, Any]] = []
    norm_query = _normalize(source_query)

    for _, row in tm_df.iterrows():
        norm_seg = _normalize(row["source_segment"])
        lev = _lev_ratio(norm_query, norm_seg)
        tsr = _token_sort_ratio(source_query, row["source_segment"])
        score = max(lev, tsr)

        if score < threshold:
            continue

        context_matched = bool(
            context_before is not None
            and _normalize(context_before) == _normalize(str(row.get("prev_source", "")))
        )

        results.append({
            "tuid": row["tuid"],
            "source_segment": row["source_segment"],
            "target_segment": row["target_segment"],
            "score": score,
            "lev_ratio": lev,
            "token_sort_ratio": tsr,
            "match_type": _classify(score, context_matched),
        })

    results.sort(key=lambda item: (item["score"], item["tuid"]), reverse=True)
    return results[:top_n]


def apply_term_glossary(text: str, glossary_df: pd.DataFrame,
                        template: str = "{source} [{target}]") -> str:
    """Разметить исходный текст переводными эквивалентами из глоссария.

    Требования: поиск регистронезависимый, но строго по границам слов; более
    длинные термины обрабатываются раньше коротких; уже подставленные фрагменты
    повторно не размечаются.

    Args:
        text: исходный сегмент.
        glossary_df: таблица глоссария с колонками ``source_term``/``target_term``.
        template: шаблон подстановки с полями ``{source}`` и ``{target}``.

    Returns:
        Текст с размеченными терминами; регистр исходного текста сохраняется.
        Если глоссарий пуст или текст пустой — возвращается исходный текст.
    """
    if glossary_df is None or glossary_df.empty or not text:
        return text

    # Сначала длинные термины — чтобы «continuous integration» сработал
    # раньше, чем «integration».
    pairs = sorted(
        zip(glossary_df["source_term"], glossary_df["target_term"]),
        key=lambda pair: len(pair[0]),
        reverse=True,
    )

    protected: List[str] = []

    def _store(replacement: str) -> str:
        """Временно спрятать подстановку, чтобы не обрабатывать её повторно."""
        protected.append(replacement)
        return f"\x00{len(protected) - 1}\x00"

    result = text
    for source_term, target_term in pairs:
        pattern = re.compile(
            rf"(?<!\w){re.escape(source_term)}(?!\w)",
            re.IGNORECASE,
        )
        result = pattern.sub(
            lambda m: _store(template.format(source=m.group(0), target=target_term)),
            result,
        )

    return re.sub(r"\x00(\d+)\x00", lambda m: protected[int(m.group(1))], result)

### 4.1. Вспомогательные функции (реализуйте при необходимости)

Эти функции не оцениваются отдельно, но нужны для `calculate_fuzzy_match`.

In [27]:
def normalize(text: str) -> str:
    """Нормализовать строку: NFKC, нижний регистр, без пунктуации, одиночные пробелы."""
    text = unicodedata.normalize("NFKC", text).lower().replace("ё", "е")
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()


def levenshtein_ratio(a: str, b: str) -> float:
    """Процент сходства строк на основе расстояния Левенштейна.

    Формула: ``(1 - distance / max(len(a), len(b))) * 100``.
    Для двух пустых строк вернуть 100.0.
    """
    if not a and not b:
        return 100.0
    longest = max(len(a), len(b))
    return round((1.0 - edit_distance(a, b) / longest) * 100.0, 2)


def token_sort_ratio(a: str, b: str) -> float:
    """Процент сходства при игнорировании порядка слов.

    Нормализовать обе строки, разбить на токены, отсортировать, склеить обратно
    и сравнить через :func:`levenshtein_ratio`.
    """
    sorted_a = " ".join(sorted(normalize(a).split()))
    sorted_b = " ".join(sorted(normalize(b).split()))
    return levenshtein_ratio(sorted_a, sorted_b)

## Блок 5. Автотесты

Запустите ячейки ниже **после** реализации функций. Все проверки должны
завершиться без `AssertionError`. Тесты работают на демо-файлах из Блока 3.

In [28]:
# --- Тест 1: метрики сходства -------------------------------------------
assert abs(levenshtein_ratio("telemetry", "telemetry") - 100.0) < 1e-6, \
    "Одинаковые строки должны давать 100 %"
assert levenshtein_ratio("", "") == 100.0, "Две пустые строки — 100 %"
assert 0.0 <= levenshtein_ratio("payload", "airframe") < 50.0, \
    "Непохожие строки должны давать низкий процент"
assert abs(token_sort_ratio("angle of attack", "attack of angle") - 100.0) < 1e-6, \
    "Token Sort Ratio должен игнорировать порядок слов"
assert normalize("  Angle   OF ATTACK, critical! ") == "angle of attack critical", \
    "normalize: нижний регистр, без пунктуации, одиночные пробелы"
print("Тест 1 (метрики сходства) — OK")

Тест 1 (метрики сходства) — OK


In [29]:
# --- Тест 2: парсинг TBX -------------------------------------------------
glossary_df = parse_tbx("demo.tbx")

assert isinstance(glossary_df, pd.DataFrame), "parse_tbx должен вернуть DataFrame"
assert len(glossary_df) == 3, f"Ожидается 3 термина, получено {len(glossary_df)}"
for column in ["entry_id", "source_term", "target_term", "pos",
               "definition", "subject_field"]:
    assert column in glossary_df.columns, f"Нет колонки '{column}'"

row = glossary_df.loc[glossary_df["source_term"] == "flight controller"].iloc[0]
assert row["target_term"] == "полётный контроллер", "Неверный русский эквивалент"
assert row["entry_id"] == "tid-001", "Не извлечён атрибут id у termEntry"
assert row["pos"] == "noun", "Не извлечён termNote[@type='partOfSpeech']"
assert "telemetry link" in set(glossary_df["source_term"]), \
    "Термин из <tig> без termNote должен быть извлечён"
print("Тест 2 (парсинг TBX) — OK")
glossary_df.head()

Тест 2 (парсинг TBX) — OK


,entry_id,source_term,target_term,pos,definition,subject_field
0,tid-001,flight controller,полётный контроллер,noun,Onboard flight computer.,Demo
1,tid-002,payload,целевая нагрузка,noun,,Demo
2,tid-003,telemetry link,канал телеметрии,,,Demo


In [30]:
# --- Тест 3: парсинг TMX -------------------------------------------------
tm_df = parse_tmx("demo.tmx")

assert isinstance(tm_df, pd.DataFrame), "parse_tmx должен вернуть DataFrame"
assert len(tm_df) == 3, f"Ожидается 3 единицы перевода, получено {len(tm_df)}"
for column in ["tuid", "source_segment", "target_segment", "domain",
               "src_len", "prev_source"]:
    assert column in tm_df.columns, f"Нет колонки '{column}'"

first = tm_df.iloc[0]
assert first["source_segment"].startswith("The flight controller"), \
    "Не извлечён текст <seg> исходного языка"
assert first["target_segment"].startswith("Полётный контроллер"), \
    "Не извлечён текст <seg> языка перевода"
assert first["prev_source"] == "", "У первой строки prev_source должен быть пустым"
assert tm_df.iloc[1]["prev_source"] == first["source_segment"], \
    "prev_source должен содержать предыдущий исходный сегмент"
assert first["src_len"] == len(first["source_segment"]), "Неверная длина сегмента"
print("Тест 3 (парсинг TMX) — OK")
tm_df.head()

Тест 3 (парсинг TMX) — OK


,tuid,source_segment,target_segment,domain,src_len,prev_source
0,1,The flight controller stabilises the aircraft.,Полётный контроллер стабилизирует аппарат.,Demo,46,
1,2,The operator checks the telemetry link.,Оператор проверяет канал телеметрии.,Demo,39,The flight controller stabilises the aircraft.
2,3,The payload is mounted under the fuselage.,Целевая нагрузка устанавливается под фюзеляжем.,Demo,42,The operator checks the telemetry link.


In [31]:
# --- Тест 4: Fuzzy Match и ICE ------------------------------------------
exact_query = "The operator checks the telemetry link."
matches = calculate_fuzzy_match(exact_query, tm_df, threshold=75.0)
assert matches, "Точное совпадение обязано быть найдено"
assert abs(matches[0]["score"] - 100.0) < 1e-6, "Точное совпадение — 100 %"
assert matches[0]["match_type"] == "Exact (100%)", \
    "Без контекста 100 % — это Exact, а не ICE"

ice = calculate_fuzzy_match(
    exact_query, tm_df, threshold=70.0,
    context_before="The flight controller stabilises the aircraft.")
assert ice[0]["match_type"] == "ICE (101%)", \
    "При совпадении предыдущего сегмента 100 % должно стать ICE"

fuzzy = calculate_fuzzy_match("The engineer checks the telemetry link.",
                              tm_df, threshold=70.0)
assert fuzzy, "Похожий сегмент должен попадать в выдачу"
assert 75.0 <= fuzzy[0]["score"] < 100.0, \
    f"Ожидается Fuzzy 75–99 %, получено {fuzzy[0]['score']}"
assert fuzzy[0]["match_type"] in {"Fuzzy", "High Fuzzy"}, "Неверный тип совпадения"

scores = [m["score"] for m in calculate_fuzzy_match("telemetry link check",
                                                    tm_df, threshold=0.0)]
assert scores == sorted(scores, reverse=True), \
    "Результаты должны быть отсортированы по убыванию сходства"
assert calculate_fuzzy_match("Совершенно посторонний текст про бухгалтерию.",
                             tm_df, threshold=75.0) == [], \
    "Ниже порога результатов быть не должно"
assert len(calculate_fuzzy_match("telemetry", tm_df, threshold=0.0, top_n=2)) <= 2, \
    "Параметр top_n не учитывается"
print("Тест 4 (Fuzzy Match и ICE) — OK")

Тест 4 (Fuzzy Match и ICE) — OK


In [32]:
# --- Тест 5: подстановка терминов ---------------------------------------
marked = apply_term_glossary("The flight controller reads the payload status.",
                             glossary_df)
assert "[полётный контроллер]" in marked, "Термин из глоссария не подставлен"
assert "[целевая нагрузка]" in marked, "Подставлен не каждый найденный термин"
assert "flight controller" in marked, "Исходный термин должен сохраняться"

assert apply_term_glossary("No terms here at all.", glossary_df) == \
    "No terms here at all.", "Текст без терминов не должен меняться"

case_test = apply_term_glossary("Payload mass is 2 kg.", glossary_df)
assert "Payload [целевая нагрузка]" in case_test, \
    "Поиск должен быть регистронезависимым, а регистр текста — сохраняться"

assert "[целевая нагрузка]" not in apply_term_glossary("payloads are heavy",
                                                       glossary_df), \
    "Совпадение должно быть по границам слов, а не по подстроке"
print("Тест 5 (подстановка терминов) — OK")
print(marked)

Тест 5 (подстановка терминов) — OK
The flight controller [полётный контроллер] reads the payload [целевая нагрузка] status.


## Блок 6. Работа со своим вариантом

После прохождения всех тестов выполните задание на данных своего варианта.

In [39]:
#6.1
def indent_tree(element, level: int = 0, space: str = "  ") -> None:
    """Рекурсивно расставить отступы в XML-дереве (работает и с lxml, и с ET)."""
    pad = "\n" + space * level
    if len(element):
        if not (element.text or "").strip():
            element.text = pad + space
        for child in element:
            indent_tree(child, level + 1, space)
        if not (element.tail or "").strip():
            element.tail = pad
        if not (element[-1].tail or "").strip():
            element[-1].tail = pad
    elif level and not (element.tail or "").strip():
        element.tail = pad

TERMS: List[Dict[str, str]] = [
    {"en": "deployment", "ru": "развертывание", "pos": "noun",
     "def_en": "An implementation, or putting into use, of something."},
    {"en": "continuous integration", "ru": "технология непрерывной интеграции", "pos": "noun",
     "def_en": "The practice of merging all branches to a shared mainline several times a day."},
    {"en": "deployment pipeline", "ru": "конвейер развертывания", "pos": "noun",
     "def_en": "An automated implementation of your application's build, deploy, test, and release process."},
    {"en": "installer", "ru": "инсталлятор", "pos": "noun",
     "def_en": "A program that installs software and prepares it for use."},
    {"en": "binary code", "ru": "бинарный код", "pos": "noun",
     "def_en": "A code that uses only the numbers 0 and 1."},
    {"en": "feedback", "ru": "обратная связь", "pos": "noun",
     "def_en": "Critical assessment of a process or activity or of their results."},
    {"en": "web server", "ru": "веб-сервер", "pos": "noun",
     "def_en": "Software that delivers web pages and other documents to browsers using the HTTP protocol."},
    {"en": "release", "ru": "релиз", "pos": "noun",
     "def_en": "The distribution of an initial or new and upgraded version of a computer software product."},
    {"en": "manually", "ru": "вручную", "pos": "adverb",
     "def_en": "By hand, using your hands."},
    {"en": "result", "ru": "результат", "pos": "noun",
     "def_en": "A positive result or consequence."},
    {"en": "documentation", "ru": "документация", "pos": "noun",
     "def_en": "Documents that explain the operation of a particular software program."},
    {"en": "development team", "ru": "команда разработчиков", "pos": "noun",
     "def_en": "Team of software development experts."},
    {"en": "automated", "ru": "автоматизированный", "pos": "adjective",
     "def_en": "Carried out by machines or computers without needing human control."},
    {"en": "tester", "ru": "тестировщик", "pos": "noun",
     "def_en": "A person or a machine that tests something."},
    {"en": "application", "ru": "приложение", "pos": "noun",
     "def_en": "A computer program that is designed for a particular purpose."},
    {"en": "version control", "ru": "управление версиями", "pos": "noun",
     "def_en": "The process of maintaining a database of all the source code and related files in a software development project to keep track of changes made during the project."},
    {"en": "URL", "ru": "URL-адрес", "pos": "noun",
     "def_en": "Abbreviation for uniform resource locator: a website address."},
    {"en": "rollback", "ru": "откат", "pos": "noun",
     "def_en": "An operation which returns a database, or group of records in a database, to a previous state."},
    {"en": "cycle time", "ru": "продолжительность цикла", "pos": "noun",
     "def_en": "The time it takes from deciding to make a change, whether a bugfix or a feature, to having it available to users."},
    {"en": "configuration", "ru": "конфигурация", "pos": "noun",
     "def_en": "The way in which all the equipment that makes up a computer system is set to operate."},
    {"en": "component", "ru": "компонент", "pos": "noun",
     "def_en": "One of the parts of a system, process, or machine."},
    {"en": "executable code", "ru": "исполняемый код", "pos": "noun",
     "def_en": "Instructions for a computer in a form the computer can directly use."},
    {"en": "source code", "ru": "исходный код", "pos": "noun",
     "def_en": "Human-readable instructions in a programming language, to be transformed into machine instructions by a compiler, assembler or other translator, or to be carried out directly by an interpreter."},
    {"en": "performance", "ru": "производительность", "pos": "noun",
     "def_en": "The total amount that can be contained or produced."},
    {"en": "customer", "ru": "заказчик", "pos": "noun",
     "def_en": "A person or an organization that buys a product or service."},
    {"en": "fault", "ru": "ошибка", "pos": "noun",
     "def_en": "Something that is wrong with a machine or system."},
    {"en": "user interface", "ru": "пользовательский интерфейс", "pos": "noun",
     "def_en": "The way in which the information on a computer, phone, etc. and instructions on how to use it are arranged on the screen and shown to the user."},
    {"en": "hardware", "ru": "оборудование", "pos": "noun",
     "def_en": "Equipment or the physical and electronic parts of a computer."},
    {"en": "lean manufacturing", "ru": "бережливое производство", "pos": "noun",
     "def_en": "The business of producing goods in large numbers using methods that avoid waste and reduce the time taken."},
    {"en": "version", "ru": "версия", "pos": "noun",
     "def_en": "Particular form of something that is slightly different from other forms of the same thing."},
    {"en": "button", "ru": "кнопка", "pos": "noun",
     "def_en": "A small, sometimes circular object that you press to operate a device or a machine, or an area on a computer screen that looks and acts like one of these."},
    {"en": "flexibility", "ru": "гибкость", "pos": "noun",
     "def_en": "The ability to change or be changed easily according to the situation."},
    {"en": "analyst", "ru": "аналитик", "pos": "noun",
     "def_en": "Someone whose job is to study or examine something in detail, in order to discover or understand more about it and often to make predictions relating to it."},
    {"en": "configuration management", "ru": "управление конфигурациями", "pos": "noun",
     "def_en": "The process by which all artifacts relevant to your project, and the relationships between them, are stored, retrieved, uniquely identified, and modified."},
    {"en": "compiler", "ru": "компилятор", "pos": "noun",
     "def_en": "A computer program that changes instructions into machine language."},
    {"en": "release candidate", "ru": "релиз-кандидат", "pos": "noun",
     "def_en": "A version of a program that is nearly ready for release but may still have a few bugs."},
    {"en": "defect", "ru": "дефект", "pos": "noun",
     "def_en": "A fault that spoils something or causes it not to work correctly."},
    {"en": "functionality", "ru": "функциональность", "pos": "noun",
     "def_en": "Any or all of the operations performed by a piece of equipment or a software program."},
    {"en": "collaborate", "ru": "сотрудничать", "pos": "verb",
     "def_en": "To work together with others to achieve a common goal."},
    {"en": "fundamentals", "ru": "основы", "pos": "noun",
     "def_en": "The main or most important rules or parts."},
    {"en": "stack", "ru": "стек", "pos": "noun",
     "def_en": "A way of storing data on a computer so that the last piece of data to be stored is the first one to be found by the computer."},
    {"en": "snapshot", "ru": "снимок", "pos": "noun",
     "def_en": "A file or set of files captured at a particular time, capable of being reloaded to restore the earlier state."},
    {"en": "settings", "ru": "параметры", "pos": "noun",
     "def_en": "Personal configuration options that allow users to customize their applications or operating system."},
    {"en": "store", "ru": "хранить", "pos": "verb",
     "def_en": "To put or keep something somewhere until it is needed."},
    {"en": "procedures", "ru": "процедуры", "pos": "noun",
     "def_en": "Set of instructions in a computer program that does a particular task."},
    {"en": "project managers", "ru": "менеджеры проекта", "pos": "noun",
     "def_en": "Someone whose job is to plan a piece of work or activity and organize the work of all the people involved in it."},
    {"en": "file", "ru": "файл", "pos": "noun",
     "def_en": "Information stored on a computer as one unit with one name."},
    {"en": "subsystems", "ru": "подсистемы", "pos": "noun",
     "def_en": "A system that forms part of a larger system."},
    {"en": "virtual machines", "ru": "виртуальные машины", "pos": "noun",
     "def_en": "Computer system that is created by software on one computer to copy the operations performed by a separate computer."},
    {"en": "delete", "ru": "удалять", "pos": "verb",
     "def_en": "To remove data, documents, programs, etc. from a computer."},
    {"en": "branch", "ru": "ветвь", "pos": "noun",
     "def_en": "A part of something larger."},
    {"en": "description", "ru": "описание", "pos": "noun",
     "def_en": "Something that tells you what something or someone is like."},
    {"en": "debugging", "ru": "отладка", "pos": "noun",
     "def_en": "To remove bugs or mistakes from a computer program."},
    {"en": "libraries", "ru": "библиотеки", "pos": "noun",
     "def_en": "A collection of subprograms used to develop software."},
    {"en": "modules", "ru": "модули", "pos": "noun",
     "def_en": "A section of a program, a subroutine."},
    {"en": "copies", "ru": "копии", "pos": "noun",
     "def_en": "Something that is exactly like another thing."},
    {"en": "interpreted language", "ru": "интерпретируемый язык", "pos": "noun",
     "def_en": "Programming language for which most of its implementations execute instructions directly."},
    {"en": "external", "ru": "внешние", "pos": "adjective",
     "def_en": "Outside of something, on the exterior."},
    {"en": "correlate", "ru": "согласовывать", "pos": "verb",
     "def_en": "To compare things and bring them into a relation having corresponding characteristics."},
    {"en": "split", "ru": "разбить", "pos": "verb",
     "def_en": "To divide into two or more parts."},
    {"en": "recompilation", "ru": "перекомпиляция", "pos": "noun",
     "def_en": "Compiling a computer program for a second, third, etc. time."},
    {"en": "run time", "ru": "время выполнения", "pos": "noun",
     "def_en": "The time during which a program is executing."},
    {"en": "pitfalls", "ru": "ловушки", "pos": "noun",
     "def_en": "Unexpected danger or difficulty."},
    {"en": "programming languages", "ru": "языки программирования", "pos": "noun",
     "def_en": "A set of rules, words, etc. that are used for writing computer programs."},
    {"en": "directory", "ru": "каталог", "pos": "noun",
     "def_en": "A structured listing of the names and characteristics of the files on a storage device."},
    {"en": "repository", "ru": "хранилище", "pos": "noun",
     "def_en": "A place where something is deposited, as for storage, safekeeping, or preservation."},
    {"en": "filesystem", "ru": "файловая система", "pos": "noun",
     "def_en": "A method of organizing blocks on a storage device into files and directories."},
    {"en": "digital certificates", "ru": "цифровые сертификаты", "pos": "noun",
     "def_en": "Digital documents that are commonly used for authentication and to help secure information on a network."},
    {"en": "encrypted", "ru": "зашифрованный", "pos": "adjective",
     "def_en": "Converted into a coded form."},
    {"en": "authentication", "ru": "аутентификация", "pos": "noun",
     "def_en": "Process of verifying identity or validity."},
    {"en": "remotely", "ru": "удаленно", "pos": "adverb",
     "def_en": "From a distance."},
    {"en": "change history", "ru": "история изменений", "pos": "noun",
     "def_en": "Information that is maintained about changes made in past editing sessions."},
    {"en": "on demand", "ru": "по требованию", "pos": "adverb",
     "def_en": "Whenever needed or requested."},
    {"en": "cluster", "ru": "кластер", "pos": "noun",
     "def_en": "A group of computers connected together to work as one larger computer and to improve their performance."},
    {"en": "record", "ru": "запись", "pos": "noun",
     "def_en": "A single row in a relational database."},
    {"en": "smoke tests", "ru": "дымовые тесты", "pos": "noun",
     "def_en": "Basic test to ensure a system won't fail on start."},
    {"en": "installation", "ru": "инсталляция", "pos": "noun",
     "def_en": "Act of setting up equipment or software."},
    {"en": "lifecycle", "ru": "жизненный цикл", "pos": "noun",
     "def_en": "Series of developments from beginning to end of usefulness."},
    {"en": "infrastructure", "ru": "инфраструктура", "pos": "noun",
     "def_en": "The equipment, software, etc. that a computer system needs in order to operate and communicate with other computers."},
    {"en": "file descriptors", "ru": "дескрипторы", "pos": "noun",
     "def_en": "Stored information that names or identifies something in a system."},
    {"en": "performance", "ru": "производительность", "pos": "noun",
     "def_en": "The amount of useful work accomplished by a computer system compared to the time and resources used."},
    {"en": "environment", "ru": "среда", "pos": "noun",
     "def_en": "The system in which a computer or computer program operates."},
    {"en": "software packages", "ru": "пакеты ПО", "pos": "noun",
     "def_en": "Suite of programs working together for tasks."},
    {"en": "topology", "ru": "топология", "pos": "noun",
     "def_en": "Arrangement of nodes in a communications network."},
    {"en": "baseline", "ru": "базовая среда", "pos": "noun",
     "def_en": "An environment that is in a properly deployed state."},
    {"en": "from scratch", "ru": "с нуля", "pos": "adverb",
     "def_en": "From the beginning."},
    {"en": "working state", "ru": "работоспособное состояние", "pos": "noun",
     "def_en": "Good condition or operational state."},
    {"en": "command line", "ru": "командная строка", "pos": "noun",
     "def_en": "Interface for typing text commands to a computer."},
    {"en": "codebase", "ru": "кодовая база", "pos": "noun",
     "def_en": "The body of source code (and possibly auxiliary data files) used to create a particular computer program."},
    {"en": "unit tests", "ru": "модульные тесты", "pos": "noun",
     "def_en": "Tests of a single component of software."},
    {"en": "network", "ru": "сеть", "pos": "noun",
     "def_en": "Number of computers that are connected together so that they can share information."},
    {"en": "component tests", "ru": "тесты компонентов", "pos": "noun",
     "def_en": "Tests of the behavior of several components of the application."},
    {"en": "colleagues", "ru": "коллеги", "pos": "noun",
     "def_en": "People you work with in a job."},
    {"en": "revert", "ru": "отменить", "pos": "verb",
     "def_en": "Return to a previous state or condition."},
    {"en": "extreme programming", "ru": "экстремальное программирование", "pos": "noun",
     "def_en": "A software engineering methodology prescribing a set of daily stakeholder practices taken to extreme levels."},
    {"en": "refactoring", "ru": "рефакторинг", "pos": "noun",
     "def_en": "Making a series of small, incremental changes that improve your code without changing your application's behavior."},
    {"en": "architecture", "ru": "архитектура", "pos": "noun",
     "def_en": "Structure and organization of computer systems."},
    {"en": "productivity", "ru": "продуктивность", "pos": "noun",
     "def_en": "State of being efficient and productive."},
    {"en": "reports", "ru": "отчеты", "pos": "noun",
     "def_en": "A spoken or written description of an event or situation."},
    {"en": "workload", "ru": "нагрузка", "pos": "noun",
     "def_en": "The amount of work assigned to a particular worker."},
]

SUBJECT_FIELD = "Информационные технологии: разработка ПО и DevOps"
TBX_PATH = "variant_1.tbx"

def build_tbx(terms: Sequence[Dict[str, str]], path: str,
              subject_field: str = SUBJECT_FIELD) -> str:
    """Сгенерировать валидный TBX-файл (TBX-Core v2, martif)."""
    martif = ET.Element("martif", {"type": "TBX", XML_LANG: "en"})

    header = ET.SubElement(martif, "martifHeader")
    file_desc = ET.SubElement(header, "fileDesc")
    title_stmt = ET.SubElement(file_desc, "titleStmt")
    ET.SubElement(title_stmt, "title").text = f"Glossary EN-RU: {subject_field}"
    source_desc = ET.SubElement(file_desc, "sourceDesc")
    ET.SubElement(source_desc, "p").text = "Lab 01, variant 1, generated automatically"
    encoding_desc = ET.SubElement(header, "encodingDesc")
    ET.SubElement(encoding_desc, "p", {"type": "XCSURI"}).text = "TBXXCSV02.xcs"

    body = ET.SubElement(ET.SubElement(martif, "text"), "body")

    for index, item in enumerate(terms, start=1):
        entry = ET.SubElement(body, "termEntry", {"id": f"tid-{index:03d}"})
        ET.SubElement(entry, "descrip", {"type": "subjectField"}).text = subject_field

        # --- английская секция: краткая форма <tig> ---
        lang_en = ET.SubElement(entry, "langSet", {XML_LANG: "en"})
        tig = ET.SubElement(lang_en, "tig")
        ET.SubElement(tig, "term").text = item["en"]
        ET.SubElement(tig, "termNote", {"type": "partOfSpeech"}).text = item["pos"]
        ET.SubElement(tig, "descrip", {"type": "definition"}).text = item["def_en"]

        # --- русская секция: расширенная форма <ntig><termGrp> ---
        lang_ru = ET.SubElement(entry, "langSet", {XML_LANG: "ru"})
        term_grp = ET.SubElement(ET.SubElement(lang_ru, "ntig"), "termGrp")
        ET.SubElement(term_grp, "term").text = item["ru"]
        ET.SubElement(term_grp, "termNote", {"type": "partOfSpeech"}).text = item["pos"]

    indent_tree(martif)
    xml_body = ET.tostring(martif, encoding="unicode")
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write('<!DOCTYPE martif SYSTEM "TBXcoreStructV02.dtd">\n')
        handler.write(xml_body)
    return path


build_tbx(TERMS, TBX_PATH)
print(f"Создан {TBX_PATH}: {os.path.getsize(TBX_PATH)} байт, терминов: {len(TERMS)}\n")
print("--- первые 26 строк файла ---")
with open(TBX_PATH, encoding="utf-8") as handler:
    for line in list(handler)[:26]:
        print(line.rstrip())

Создан variant_1.tbx: 73137 байт, терминов: 100

--- первые 26 строк файла ---
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE martif SYSTEM "TBXcoreStructV02.dtd">
<martif type="TBX" xml:lang="en">
  <martifHeader>
    <fileDesc>
      <titleStmt>
        <title>Glossary EN-RU: Информационные технологии: разработка ПО и DevOps</title>
      </titleStmt>
      <sourceDesc>
        <p>Lab 01, variant 1, generated automatically</p>
      </sourceDesc>
    </fileDesc>
    <encodingDesc>
      <p type="XCSURI">TBXXCSV02.xcs</p>
    </encodingDesc>
  </martifHeader>
  <text>
    <body>
      <termEntry id="tid-001">
        <descrip type="subjectField">Информационные технологии: разработка ПО и DevOps</descrip>
        <langSet xml:lang="en">
          <tig>
            <term>deployment</term>
            <termNote type="partOfSpeech">noun</termNote>
            <descrip type="definition">An implementation, or putting into use, of something.</descrip>
          </tig>


In [44]:
#6.2
PARALLEL_CORPUS: List[Tuple[str, str]] = [
    ("This regular deployment gave them many advantages.",
     "Регулярное развертывание предоставляло им ряд преимуществ."),
    ("The foundation for the approach, at least for the development team, is Continuous Integration (CI).",
     "Фундаментом непрерывного развертывания (как минимум для команды разработчиков) служит концепция непрерывной интеграции (Continuous Integration – CI)."),
    ("Every organization will have differences in the implementation of their deployment pipelines, depending on their value stream for releasing software, but the principles that govern them do not vary.",
     "Конвейеры развертывания могут быть организованы по-разному, в зависимости от технологических особенностей поставки релизов, однако фундаментальные принципы непрерывного развертывания одинаковы в любых ситуациях."),
    ("One of the first steps in the pipeline is to create binaries and installers.",
     "Начальная стадия конвейера – создание бинарных кодов и инсталляторов."),
    ("Each test that the release candidate passes gives us more confidence that this particular combination of binary code, configuration information, environment, and data will work.",
     "Каждый тест увеличивает вероятность того, что текущая комбинация бинарных кодов, конфигураций, рабочей среды и данных будет работать безупречно."),
    ("Second, it improves feedback so that problems are identified, and so resolved, as early in the process as possible.",
     "Во-вторых, конвейер улучшает обратную связь, делая каждую проблему идентифицируемой и решаемой на возможно более ранней стадии процесса."),
    ("Configuration information is copied or created through the admin consoles of web servers, applications servers, or other third-party components of the system.",
     "Конфигурационная информация копируется или создается администраторами веб-серверов, серверов приложений и сторонних компонентов."),
    ("The day of a software release tends to be a tense one.",
     "Обычно день поставки релиза самый напряженный день всего рабочего процесса."),
    ("Many organizations release software manually.",
     "Многие организации развертывают ПО вручную."),
    ("Even if this is not the case, differences in the ordering and timing of these steps can lead to different outcomes.",
     "И даже если не произойдет ошибок, различия в принципах управления разными стадиями приводят к нестабильным результатам."),
    ("The production of extensive, detailed documentation that describes the steps to be taken and the ways in which the steps may go wrong.",
     "Продуцирование большого объема документации, в которой подробно описывается, что нужно сделать и что может получиться неправильно."),
    ("Frequent calls to the development team to explain why a deployment is going wrong on a release day.",
     "Частые обращения к команде разработчиков для получения объяснений, почему в день поставки релиза приложение работает не так, как предполагается."),
    ("Releasing packaged software should involve a single automated process.",
     "Выпуск пакета ПО должен быть одним автоматизированным процессом."),
    ("If testers have been involved in the process up to this point, they have tested the system on development machines.",
     "Если в процессе разработки ПО участвовали тестировщики, они уже протестировали систему в среде разработки."),
    ("When working on a new application, the first deployment to staging is likely to be the most troublesome.",
     "При работе с новым приложением больше всего проблем возникает при первом развертывании в отладочной среде."),
    ("Changes should be made in version control and then propagated to production through an automated process.",
     "Изменения должны выполняться посредством системы управления версиями, а затем автоматически распространяться на рабочую среду."),
    ("Almost every release involves last-minute changes, such as fixing the database login details or updating the URL for an external service.",
     "Почти в каждый релиз в последнюю минуту вносятся небольшие изменения, такие как исправление процедуры регистрации в базе данных, обновление URL-адреса внешней службы и т.п."),
    ("It should be possible to use the same automated process to roll back to a previous version of production if the deployment goes wrong.",
     "Если развертывание прошло неудачно, этот же автоматический процесс должен предоставлять возможность отката к предыдущей версии рабочей среды."),
    ("It is therefore vital to minimize cycle time so that an effective feedback loop can be established.",
     "Следовательно, очень важно минимизировать продолжительность цикла, чтобы существовала эффективная обратная связь между этапами разработки и применения ПО."),
    ("A working software application can be usefully decomposed into four components: executable code, configuration, host environment, and data.",
     "Работающее приложение иногда полезно условно разделить на четыре компонента: исполняемый код, конфигурация, рабочая среда и данные."),
    ("Therefore, we need to keep all four of these components under control and ensure that a change in any one of them is verified.",
     "Следовательно, нужно контролировать все четыре компонента и обеспечить возможность верификации при каждом изменении любого из них."),
    ("Executable code changes when a change is made to the source code.",
     "Исполняемый код изменяется вместе с исходным кодом."),
    ("Every time a change is made to the source code, the resulting binary must be built and tested.",
     "При каждом изменении в исходном коде результирующий двоичный код нужно заново скомпоновать и протестировать."),
    ("This checks that the application performs sufficiently well in terms of capacity, availability, security, and so on to meet its users' needs.",
     "Это означает, что тест должен проверить производительность, доступность, безопасность и другие параметры приложения, не связанные напрямую с деловой логикой."),
    ("The software must go through exploratory testing and a demonstration to the customer and a selection of users.",
     "Приложение должно быть подвергнуто исследовательскому тестированию и продемонстрировано заказчику и группе пользователей."),
    ("If any of them fails, it means our application has a critical fault and should not be released under any circumstances.",
     "Если любой тест терпит крах, значит, приложение содержит критическую ошибку и не должно быть пропущено на стадию выпуска."),
    ("That means that a test to check the color of a UI element should not be included in this set of tests.",
     "Это означает, например, что в данный набор не должно быть включено тестирование цвета какого-либо элемента пользовательского интерфейса."),
    ("This organization of the testing process means that we have a high level of confidence in the software after the first set of tests, which run fastest on the cheapest hardware.",
     "Такая схема тестирования обеспечивает достаточно высокий уровень проверки уже на первых стадиях тестов, когда они выполняются быстро и на дешевом оборудовании."),
    ("The goals of lean manufacturing are to ensure the rapid delivery of high-quality products, focusing on the removal of waste and the reduction of cost.",
     "Цель бережливого производства быстрая поставка высококачественного продукта с акцентом на удалении лишних компонентов и минимизации стоимости."),
    ("The ability to easily deploy any version of the software into any environment has many advantages.",
     "Возможность легко развернуть любую версию в любой среде предоставляет ряд преимуществ."),
    ("Releases can be performed at the push of a button.",
     "Новые версии устанавливаются путем щелчка на кнопке."),
    ("The flexibility that our deployment tools offer to them changes the way that they work – for the better.",
     "Гибкость, предоставляемая инструментами развертывания, повышает качество работы."),
    ("The analysts who capture the requirements can misunderstand.",
     "Аналитики, формулирующие требования к продукту, могут неправильно понимать их."),
    ("In our experience, this dependence on manual configuration management is common.",
     "Зависимость проектов от ручного управления конфигурациями встречается довольно часто."),
    ("Even the configuration of the compiler may change the behavior of the application.",
     "Даже конфигурация компилятора может изменить поведение приложения."),
    ("Most approaches to releasing software identify release candidates at the end of the process.",
     "В большинстве концепций развертывания ПО релиз-кандидат определяется в конце процесса."),
    ("If the resulting product is found to be free of defects, and it meets the acceptance criteria set out by the customer, then it can be released.",
     "Результирующий продукт может стать релизом, если он свободен от дефектов и удовлетворяет приемочным критериям, установленным заказчиком."),
    ("The developers have forgotten what they were doing at the time when they introduced the defect, and the functionality may have changed in the meantime.",
     "Разработчики не помнят, что они делали и какими соображениями руководствовались в момент, когда был введен дефект, а функциональность со временем может измениться."),
    ("Second, it allows teams that may be distributed across space and time to collaborate.",
     "Во-вторых, она предоставляет возможность сотрудничать друг с другом командам, вовлеченным в процесс развертывания и разделенным в пространстве и времени."),
    ("These are the fundamentals of version control.",
     "Это основы управления версиями."),
    ("This should include configuration information for your application's software stack and the operating systems.",
     "Сюда относится конфигурационная информация стека приложений, операционные системы."),
    ("This allows you to recover an exact snapshot of the state of the entire system, from development environment to production environment, at any point in the project's history.",
     "Это позволит воссоздать точный снимок состояния всей системы, от среды разработки до рабочей среды на любом этапе проекта."),
    ("It is even helpful to keep the configuration files for the development team's development environments in version control since it makes it easy for everyone on the team to use the same settings.",
     "Полезно также хранить в системе управления версиями конфигурационные файлы среды разработки, чтобы облегчить каждой команде извлечение и применение унифицированных конфигурационных параметров."),
    ("Analysts should store requirements documents.",
     "Аналитики должны хранить документацию, определяющую требования к системе."),
    ("Testers should keep their test scripts and procedures in version control.",
     "Тестировщики должны хранить свои процедуры и сценарии тестирования."),
    ("Project managers should save their release plans, progress charts, and risk logs here.",
     "Менеджеры проекта должны сохранять свои планы выпуска версий, графики хода работ и журналы рисков."),
    ("Every member of the team should store any document or file related to the project in version control.",
     "Каждый член команды должен сохранять любой документ или файл, связанный с проектом, в системе управления версиями."),
    ("The subsystems that each team was working on communicated with one another using a proprietary message protocol via IBM MQSeries.",
     "Подсистемы, над которыми работала каждая команда, сообщались друг с другом посредством патентованного протокола сообщений IBM MQSeries."),
    ("In addition to storing source code and configuration information, many projects also store binary images of their application servers, compilers, virtual machines, and other parts of their toolchain in version control.",
     "Кроме исходных кодов и конфигурационной информации в системах управления версиями хранятся двоичные образы серверов приложений, компиляторов, виртуальных машин и другие инструменты."),
    ("This freedom to delete is in itself a significant step forward in the maintainability of a large configuration set.",
     "Свобода удалять ненужное — уже огромный шаг вперед в направлении лучшей управляемости больших наборов конфигураций."),
    ("A solution that some people use to resolve this dilemma is to create a separate branch within the version control system for new functionality.",
     "Для решения этой дилеммы можно создать в системе управления версиями отдельную ветвь для новых функций."),
    ("Every version control system has the facility to add a description to your commit.",
     "Каждая система управления версиями предоставляет возможность добавлять описание при фиксации изменения."),
    ("That change can save you hours of debugging.",
     "Оно сэкономит много часов, потраченных на отладку кода."),
    ("Libraries are typically deployed in the form of binary files, are never changed by your application's development team.",
     "Библиотеки обычно развертываются в виде двоичных файлов и никогда не изменяются командой разработчиков."),
    ("Components and modules are typically under active development by other teams and change quite frequently.",
     "Компоненты и модули обычно находятся на стадии активной разработки другими командами и часто изменяются."),
    ("We recommend that you keep copies of your external libraries somewhere locally.",
     "Мы рекомендуем хранить копии внешних библиотек локально."),
    ("External libraries usually come in binary form, unless you're using an interpreted language.",
     "Обычно внешние библиотеки поставляются в двоичном виде, за исключением случаев, когда используются интерпретируемые языки."),
    ("External libraries are normally installed globally on your system by a package management system such as Ruby Gems or Perl modules.",
     "Внешние библиотеки, как правило, устанавливаются в систему глобально с помощью, например, пакетов Ruby Gems или модулей Perl."),
    ("It makes it much easier to correlate versions of your software with the versions of the libraries that were used to build them.",
     "Если хранить их, то будет значительно легче согласовывать версии приложения с версиями библиотек, используемых для сборки."),
    ("It's good practice to split all but the smallest applications into components.",
     "Рекомендуется разбить каждое приложение (кроме самых маленьких) на отдельные компоненты."),
    ("Recompiling dependencies is not only less efficient; it also means you're creating an artifact that is potentially different from the one that you already tested.",
     "Перекомпиляция зависимостей не только малоэффективна, она также приводит к появлению компонентов, отличающихся от тех, которые уже протестированы."),
    ("Configuration information can be used to change the behavior of software at build time, deploy time, and run time.",
     "Конфигурационную информацию можно использовать для управления поведением программы на этапах компиляции, развертывания и выполнения."),
    ("There are many significant pitfalls on the road to highly configurable software, but perhaps the worst are the following.",
     "На пути к повышенной конфигурируемости приложения есть много ловушек, но самые коварные, как нам кажется, следующие."),
    ("Modern computer languages have evolved all sorts of characteristics and techniques to help them reduce errors.",
     "Современные языки программирования обладают богатым набором средств, помогающих уменьшить количество ошибок."),
    ("There are a few obvious choices for where to store your application configuration: a database, a version control system, or a directory or registry.",
     "Существует всего несколько очевидных вариантов хранения конфигурационной информации приложения: база данных, система управления версиями и реестр (или каталог)."),
    ("It is worth keeping a list of the available configuration options for your application in the same repository as its source code.",
     "Рекомендуется хранить список доступных конфигурационных параметров приложения в том же хранилище, в котором находится исходный код."),
    ("Your application can access its configuration via a file on its local filesystem.",
     "Приложение может иметь доступ к конфигурации посредством файла в локальной файловой системе."),
    ("This separation is particularly relevant for security-related configuration elements, such as passwords and digital certificates, to which access should be restricted.",
     "Раздельное хранение особенно полезно для конфигурационных параметров, связанных с безопасностью, например паролей и цифровых сертификатов, доступ к которым должен быть ограничен."),
    ("If you must store your password somewhere other than the inside of your head, you could try putting them in your home directory in an encrypted form.",
     "Если уж нужно сохранить где-нибудь пароль, сохраните его в своем домашнем каталоге в зашифрованном виде."),
    ("There are several acceptable ways to handle authentication for a multilayer system.",
     "В многослойных системах существует несколько допустимых способов аутентификации."),
    ("Databases, directories, and registries are convenient places to store configuration since they can be accessed remotely.",
     "Базы данных, каталоги и реестры удобные места для хранения конфигурационной информации, потому что они поддерживают удаленный доступ."),
    ("However, make sure to keep the history of changes to configuration for the purposes of audit and rollback.",
     "Однако нужно хранить историю изменений конфигураций для целей аудита и откатов."),
    ("Have a script that loads the appropriate version into your database or directory on demand.",
     "Создайте сценарий, загружающий соответствующую версию в базу данных или каталог по требованию."),
    ("There is also the problem of keeping configuration on filesystems in sync if, for example, you need to run your application on a cluster.",
     "Кроме того, в этом случае нужно решить проблемы синхронизации конфигураций в разных файловых системах, например когда приложение выполняется в кластере."),
    ("Each configuration setting can be modeled as a tuple.",
     "Каждый конфигурационный параметр может быть представлен как запись в базе данных."),
    ("The second stage is to actually run some smoke tests once your application is installed to make sure it is operating as expected.",
     "Вторая стадия – фактическое выполнение дымовых тестов, когда приложение установлено."),
    ("Ideally, these tests should stop the application and fail the installation or deployment process if the results are not as expected.",
     "В идеале каждый такой тест должен остановить приложение и отменить процесс инсталляции или развертывания, если результаты отличаются от ожидаемых."),
    ("Consider where in your application's lifecycle it makes sense to inject a particular piece of configuration.",
     "Проанализируйте, в какой точке жизненного цикла приложения лучше ввести в него порцию конфигурационной информации."),
    ("Every application depends on hardware, software, infrastructure, and external systems in order to work.",
     "Работа каждого приложения зависит от оборудования, программного обеспечения, инфраструктуры и внешних систем."),
    ("For example, you may have an application that relies on a large number of file descriptors being available.",
     "Например, приложение может зависеть от доступности большого количества дескрипторов файлов."),
    ("One small change can break the whole application or severely degrade its performance.",
     "Одно небольшое изменение может разрушить все приложение или существенно ухудшить его производительность."),
    ("It should always be cheaper to create a new environment than to repair an old one.",
     "Всегда дешевле создать новую среду, чем отладить старую."),
    ("The additional software packages that need to be installed on each environment to support your application, including their versions and configuration.",
     "Дополнительные пакеты ПО, которые должны быть установлены в каждой среде для поддержки приложения, включая их версии, параметры и конфигурации."),
    ("The networking topology required for your application to work.",
     "Топология сетей, необходимых для работы приложения."),
    ("Your automated environment provisioning system should be able to establish, or reestablish, any given baseline that has existed in the recent history of your project.",
     "Система автоматического создания сред должна уметь устанавливать и переустанавливать любую заданную базовую среду, существовавшую когда-либо в проекте."),
    ("Instead of creating a new environment from scratch using an automated process, you can simply take a copy of each box in your environment and store it as a baseline.",
     "Вместо создания новой среды с нуля, можно извлечь копию каждого компонента среды и сохранить ее в качестве базового компонента."),
    ("The goal of continuous integration is that the software is in a working state all the time.",
     "Главная цель непрерывной интеграции заключается в том, чтобы приложение постоянно находилось в работоспособном состоянии."),
    ("You must be able to start your build from the command line.",
     "Обязательно нужно иметь возможность запускать сборку приложения из командной строки."),
    ("Your build scripts should be treated like your codebase.",
     "Со сценариями сборки нужно работать так же, как и с кодовой базой."),
    ("However, there are three kinds of tests we are interested in running from our continuous integration build: unit tests, component tests, and acceptance tests.",
     "Однако существуют три типа тестов, интересных с точки зрения непрерывной интеграции: модульные тесты, тесты компонентов и приемочные тесты."),
    ("They do not hit the database (if your application has one), the filesystem, or the network.",
     "Они не покрывают базу данных (если она используется в приложении), файловую систему или сеть."),
    ("Component tests typically take longer to run.",
     "Тесты компонентов выполняются, как правило, дольше модульных."),
    ("All your colleagues are walking out of the door, and you have just committed your changes.",
     "Все ваши коллеги дружно собирают вещи, а вы только что зафиксировали ваши изменения."),
    ("Assume that you may break something that will take more than a few minutes, and know what to do to revert the changes and get back to the known-good revision in version control.",
     "Предполагайте, что в любой момент может произойти ошибка, на исправление которой может уйти несколько минут, и будьте готовы отменить изменение и вернуться к последней хорошей версии."),
    ("Continuous integration is one of the twelve core extreme programming practices.",
     "Непрерывная интеграция это одна из двенадцати базовых методик экстремального программирования."),
    ("CI and test-driven development enable refactoring by assuring you that your changes don't alter the existing behavior of the application.",
     "Непрерывная интеграция и разработка через тестирование способствуют рефакторингу кодов тем, что они гарантируют неизменность поведения кода."),
    ("Sometimes there are aspects of the architecture of a system that are too easy for developers to forget.",
     "Часто архитектура системы обладает некоторыми особенностями, о которых разработчики быстро забывают."),
    ("Time and time again we have seen it make a step change to the productivity of software development teams.",
     "Снова и снова мы убеждаемся в том, что это важнейший шаг в направлении повышения продуктивности команды."),
    ("A system of reference for reports and installers for your testing team.",
     "Система ссылок для инсталляторов и отчетов, необходимая команде тестировщиков."),
    ("They reduce the workload on testers.",
     "Они уменьшают нагрузку на тестировщиков."),
]

VARIANT = 1
TMX_FILE = f"variant_{VARIANT}.tmx"
DOMAIN = "Информационные технологии: разработка ПО и DevOps"

def build_tmx(pairs: Sequence[Tuple[str, str]], path: str,
              src_lang: str = "en", tgt_lang: str = "ru",
              domain: str = DOMAIN) -> str:
    """Сгенерировать валидную память переводов в формате TMX 1.4b."""
    from datetime import datetime, timezone

    created = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

    tmx = ET.Element("tmx", {"version": "1.4"})
    ET.SubElement(tmx, "header", {
        "creationtool": "lab01-generator",
        "creationtoolversion": "1.0",
        "segtype": "sentence",
        "o-tmf": "TMX",
        "adminlang": "en",
        "srclang": src_lang,
        "datatype": "plaintext",
        "creationdate": created,
    })
    body = ET.SubElement(tmx, "body")

    for index, (source, target) in enumerate(pairs, start=1):
        tu = ET.SubElement(body, "tu", {"tuid": str(index), "creationdate": created})
        ET.SubElement(tu, "prop", {"type": "domain"}).text = domain
        for lang, text in ((src_lang, source), (tgt_lang, target)):
            tuv = ET.SubElement(tu, "tuv", {XML_LANG: lang})
            ET.SubElement(tuv, "seg").text = text

    indent_tree(tmx)
    xml_body = ET.tostring(tmx, encoding="unicode")
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write('<!DOCTYPE tmx SYSTEM "tmx14.dtd">\n')
        handler.write(xml_body)
    return path

build_tmx(PARALLEL_CORPUS, TMX_FILE)
print(f"Создан {TMX_FILE}: {os.path.getsize(TMX_FILE)} байт, "
      f"единиц перевода: {len(PARALLEL_CORPUS)}\n")
print("--- первые 20 строк файла ---")
with open(TMX_FILE, encoding="utf-8") as handler:
    for line in list(handler)[:20]:
        print(line.rstrip())

Создан variant_1.tmx: 61662 байт, единиц перевода: 100

--- первые 20 строк файла ---
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE tmx SYSTEM "tmx14.dtd">
<tmx version="1.4">
  <header creationtool="lab01-generator" creationtoolversion="1.0" segtype="sentence" o-tmf="TMX" adminlang="en" srclang="en" datatype="plaintext" creationdate="20261017T140810Z"/>
  <body>
    <tu tuid="1" creationdate="20261017T140810Z">
      <prop type="domain">Информационные технологии: разработка ПО и DevOps</prop>
      <tuv xml:lang="en">
        <seg>This regular deployment gave them many advantages.</seg>
      </tuv>
      <tuv xml:lang="ru">
        <seg>Регулярное развертывание предоставляло им ряд преимуществ.</seg>
      </tuv>
    </tu>
    <tu tuid="2" creationdate="20261017T140810Z">
      <prop type="domain">Информационные технологии: разработка ПО и DevOps</prop>
      <tuv xml:lang="en">
        <seg>The foundation for the approach, at least for the development team, is Continuous Integrat

In [46]:
VARIANT = 1
SURNAME = "Asadova"

TBX_FILE = f"variant_{VARIANT}.tbx"
TMX_FILE = f"variant_{VARIANT}.tmx"

# 1) файлы уже созданы в Блоках 6.1 и 6.2
print(f"TBX : {TBX_FILE} ({os.path.getsize(TBX_FILE)} байт)")
print(f"TMX : {TMX_FILE} ({os.path.getsize(TMX_FILE)} байт)")

# 2) глоссарий
my_glossary = parse_tbx(TBX_FILE)
print(f"\nГлоссарий: {len(my_glossary)} терминов")

# 3) память переводов
my_tm = parse_tmx(TMX_FILE)
print(f"Память переводов: {len(my_tm)} сегментов")

# 4) три запроса: ICE, Fuzzy, No Match
QUERIES: List[Dict[str, Optional[str]]] = [
    {
        "label": "1. ICE — точное совпадение с совпадением контекста",
        "query": "The foundation for the approach, at least for the development team, is Continuous Integration (CI).",
        "context": "This regular deployment gave them many advantages.",
    },
    {
        "label": "2. Fuzzy — заменено одно слово",
        "query": "Many organizations release software by hand.",
        "context": None,
    },
    {
        "label": "3. No match — сегмент вне предметной области",
        "query": "The quarterly financial report was approved by the board.",
        "context": None,
    },
]

all_matches: List[Dict[str, Any]] = []
for case in QUERIES:
    matches = calculate_fuzzy_match(case["query"], my_tm, threshold=75.0,
                                    context_before=case["context"])
    print("=" * 100)
    print(case["label"])
    print(f"Запрос: {case['query']}")
    if not matches:
        print("  -> совпадений выше порога 75 % не найдено, сегмент переводится с нуля")
        continue
    for rank, match in enumerate(matches, start=1):
        print(f"  #{rank} {match['score']:6.2f} %  [{match['match_type']:<12}] "
              f"tuid={match['tuid']}")
        print(f"      TM src: {match['source_segment']}")
        print(f"      TM tgt: {match['target_segment']}")
        match["query"] = case["query"]
        all_matches.append(match)

matches_df = pd.DataFrame(all_matches)
print("=" * 100)
print(f"\nВсего кандидатов выше порога: {len(matches_df)}")

# 5) экспорт CSV для Smartcat
GLOSSARY_CSV = f"glossary_smartcat_v{VARIANT}.csv"
TM_CSV = f"tm_segments_v{VARIANT}.csv"
FUZZY_CSV = f"fuzzy_report_v{VARIANT}.csv"

smartcat_df = pd.DataFrame({
    "Source (en)": my_glossary["source_term"],
    "Target (ru)": my_glossary["target_term"],
    "Comment": my_glossary["definition"],
})
smartcat_df.to_csv(GLOSSARY_CSV, index=False, encoding="utf-8-sig")

my_tm[["tuid", "source_segment", "target_segment", "domain"]].to_csv(
    TM_CSV, index=False, encoding="utf-8-sig")

matches_df[["query", "tuid", "source_segment", "target_segment",
            "score", "lev_ratio", "token_sort_ratio", "match_type"]].to_csv(
    FUZZY_CSV, index=False, encoding="utf-8-sig")

print("\nЭкспортированные файлы:")
for name in [TBX_FILE, TMX_FILE, GLOSSARY_CSV, TM_CSV, FUZZY_CSV]:
    print(f"  {name:<28} {os.path.getsize(name):>7} байт")

# 6) итоговая таблица совпадений
print("\nИтоговая таблица совпадений:")
matches_df[["query", "tuid", "score", "match_type"]]

TBX : variant_1.tbx (73137 байт)
TMX : variant_1.tmx (61662 байт)

Глоссарий: 100 терминов
Память переводов: 100 сегментов
1. ICE — точное совпадение с совпадением контекста
Запрос: The foundation for the approach, at least for the development team, is Continuous Integration (CI).
  #1 100.00 %  [ICE (101%)  ] tuid=2
      TM src: The foundation for the approach, at least for the development team, is Continuous Integration (CI).
      TM tgt: Фундаментом непрерывного развертывания (как минимум для команды разработчиков) служит концепция непрерывной интеграции (Continuous Integration – CI).
2. Fuzzy — заменено одно слово
Запрос: Many organizations release software by hand.
  #1  84.09 %  [Fuzzy       ] tuid=9
      TM src: Many organizations release software manually.
      TM tgt: Многие организации развертывают ПО вручную.
3. No match — сегмент вне предметной области
Запрос: The quarterly financial report was approved by the board.
  -> совпадений выше порога 75 % не найдено, сегмент 

,query,tuid,score,match_type
0,"The foundation for the approach, at least for the develo...",2,100.00,ICE (101%)
1,Many organizations release software by hand.,9,84.09,Fuzzy


## Блок 7. Выводы

Ответьте письменно (3–5 предложений на пункт):

1. Чем концептуально-ориентированная организация TBX отличается от
   сегментно-ориентированной организации TMX?

TBX организован концептуально (по понятиям): в центре стоит termEntry — одна словарная статья, которая описывает конкретное понятие предметной области на разных языках. В отличие от него, TMX организован сегментно (по тексту): его основная единица — tu, которая хранит уже переведенные предложения (исходное и перевод). То есть TBX отвечает на вопрос «какой термин использовать?», а TMX — «переводили ли мы уже такое предложение?».

2. Почему `xml:lang` требует особой обработки при парсинге и как вы её решили?

Атрибут xml:lang принадлежит пространству имен XML, поэтому стандартные парсеры (как lxml или ElementTree) не находят его по обычному ключу "lang". Чтобы программа не падала с ошибкой и корректно определяла язык, была задана глобальная переменная XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang" и использована при извлечении атрибута (lang_set.get(XML_LANG)). Это позволило правильно разделять английскую и русскую секции.

3. В каких случаях Levenshtein ratio даёт заниженную оценку сходства и как это
   компенсирует Token Sort Ratio?

Levenshtein ratio вычисляет расстояние по символам, поэтому он сильно занижает оценку, если в предложениях одинаковые слова стоят в разном порядке (например, при перестановке слов местами). Token Sort Ratio компенсирует этот недостаток: перед сравнением он разбивает строки на слова, сортирует их по алфавиту и только потом считает сходство. Благодаря этому порядок слов игнорируется, и алгоритм находит совпадения даже в переставленных фразах.


4. Чем ICE (101 %) отличается от обычного 100 %-совпадения и почему это важно
   для стоимости проекта в Smartcat?

Обычное совпадение Exact (100 %) означает, что текст предложения полностью совпал с тем, что уже есть в памяти переводов, но само предложение может стоять в документе в другом окружении. Совпадение ICE (In-Context Exact) — это тот же самый текст на 100 %, но при этом еще и полностью совпадает контекст (то есть предыдущее и следующее предложения в документе точно такие же).

Для стоимости проекта в Smartcat это имеет огромное значение, потому что за ICE-совпадения переводчику платят по самой низкой ставке (обычно 10–20 % от полной стоимости). Так как предложение и его контекст полностью совпадают, переводчику не нужно вчитываться, проверять смысл или редактировать перевод — он может просто нажать «принять». Благодаря этому заказчик сильно экономит бюджет, а переводчик тратит минимум времени.

5. Какие проблемы возникли при автоматической подстановке терминов и как вы их
   решили?

Главная проблема — это вложенные подстановки: если сначала заменить короткое слово, оно может разрушить более длинный термин (например, «version» внутри «version control»). Кроме того, система могла заменять часть слова (например, «pitot» в слове «pitotless»). Чтобы это исправить, мы отсортировали термины по убыванию длины перед подстановкой. Также мы использовали регулярные выражения с проверкой границ слов (?<!\w)...(?!\w) и временное скрытие уже размеченных фрагментов с помощью спецсимволов \x00, чтобы избежать повторной обработки.



**Ваши выводы:**

В ходе лабораторной работы реализованы алгоритмы парсинга стандартов TBX и TMX с помощью Python и библиотеки lxml. Обработан парсинг пространств имен XML, извлечение параллельных сегментов, а также реализован нечеткий поиск (Fuzzy Match) с поддержкой контекстных совпадений ICE. Отдельно решена задача автоматической подстановки терминов из глоссария с приоритетом длинных фраз. Реализованные алгоритмы автоматизируют процессы работы с памятью переводов и терминологией, что повышает эффективность работы переводчика и снижает затраты в CAT-системах.


_______________________________________________

---

### Что сдать на проверку

1. Ссылка на Colab-ноутбук `lab_01_Фамилия.ipynb` (доступ «для всех по ссылке»).
2. Файлы `variant_NN.tbx`, `variant_NN.tmx`, `glossary_smartcat.csv`.
3. Отчёт с выводами и скриншотом импортированного глоссария в Smartcat.